# Log2Graphs conversion [HDFS]

1. Get entire parsed log set (total) with params
2. Split log set to sequences based on block id
3. Create a graph out of sequence with embeddings for templates (nodes) with additional parameter data
4. Assign edges with time weights


In [ ]:
import os, sys, re, json
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath(".."))

from src.parser.drain_parser import DrainParser

LOG_PATH    = "../data/raw/HDFS_full.log"
CONFIG_PATH = "../configs/drain.ini"
MAX_LINES   = None  # set an int (e.g. 500_000) for a quick preview


## Step 1 — Parse raw logs into a structured DataFrame

Run Drain on the full log file to learn templates, then do a second pass (`annotate_file`) that uses the **final** (fully generalised) templates to assign each line:
- its stable `cluster_id` and `template` string
- the `parameters` extracted for every `<*>` token
- the `block_id` (regex-extracted `blk_XXX`)
- a parsed `timestamp`

In [ ]:
# Fit Drain on the raw log (learns templates online)
parser = DrainParser(config_path=CONFIG_PATH)
parser.fit_file(LOG_PATH, max_lines=MAX_LINES)

# Second pass: annotate every line with its final cluster_id, template, params, block_id
df = parser.annotate_file(LOG_PATH, max_lines=MAX_LINES)

print(df.shape)
df.head(3)


## Step 1b — Persist annotated logs to Parquet

Save the full annotated DataFrame to `data/processed/` so subsequent runs can skip the expensive Drain fit+annotate pass.  
The `parameters` column (list of strings) is stored as a JSON-encoded string to keep the Parquet schema flat and portable.


In [ ]:
!pip install fastparquet --quiet


In [ ]:
import json
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
PARQUET_PATH  = PROCESSED_DIR / "hdfs_annotated.parquet"

# Parquet cannot natively store Python lists in all engines,
# so encode the `parameters` column as a JSON string.
df_save = df.copy()
df_save["parameters"] = df_save["parameters"].apply(json.dumps)

# fastparquet cannot write pandas Arrow-backed string columns (ArrowDtype).
# The error "Unable to avoid copy while creating an array as requested" is
# caused by those columns. Cast all string/object columns to plain numpy object.
for col in df_save.select_dtypes(include=["object", "string"]).columns:
    df_save[col] = df_save[col].astype(object)

# Use fastparquet to avoid the pandas/pyarrow version-conflict bug
# (ArrowKeyError: arrow.py_extension_type) that occurs when pyarrow
# is loaded after pandas in the same kernel session.
df_save.to_parquet(PARQUET_PATH, index=False, engine="fastparquet")

print(f"Saved {len(df_save):,} rows  →  {PARQUET_PATH}")
print(f"File size : {PARQUET_PATH.stat().st_size / 1_048_576:.1f} MB")
print(f"Columns   : {df_save.columns.tolist()}")


In [ ]:
PARSER_PATH = "../models/drain_parser.bin"

# Save — triggers drain3's native save_state("manual_save") which writes a
# compressed jsonpickle snapshot of the Drain trie (NOT a Python pickle).
parser.save(PARSER_PATH)

# ── To reload in a later session without re-fitting: ──────────────────────────
# from src.parser.drain_parser import DrainParser
#
# parser = DrainParser.load(PARSER_PATH, config_path=CONFIG_PATH)
# # TemplateMiner loads the snapshot automatically on construction.
# # Then annotate as normal:
# df = parser.annotate_file(LOG_PATH)


## Step 2 — Split into per-block sequences

Group log lines by `block_id`. Each group is an **ordered time-series** of events (template firings) for one HDFS block — the unit of anomaly labelling in the benchmark.

In [ ]:
# Drop lines where no block_id was found (e.g. NameNode bookkeeping without a block ref)
df_blocks = df.dropna(subset=["block_id"]).copy()
df_blocks = df_blocks.sort_values(["block_id", "timestamp"])

# sequences: dict[block_id -> DataFrame of ordered events]
sequences = {bid: grp.reset_index(drop=True) for bid, grp in df_blocks.groupby("block_id")}

print(f"Total lines with block_id : {len(df_blocks):,}")
print(f"Unique blocks             : {len(sequences):,}")
print(f"\nSample sequence for {list(sequences.keys())[0]}:")
sequences[list(sequences.keys())[0]][["timestamp", "cluster_id", "template", "parameters"]].head(6)


In [ ]:
import json
from pathlib import Path

SEQUENCES_PATH = Path("../data/processed/hdfs_sequences.parquet")

# ── Save ──────────────────────────────────────────────────────────────────────
# df_blocks is the flat table that sequences were built from.
# Encode the `parameters` list column as JSON strings for Parquet compatibility.
df_seq_save = df_blocks.copy()
df_seq_save["parameters"] = df_seq_save["parameters"].apply(json.dumps)

# Cast Arrow-backed string columns to plain numpy object dtype.
for col in df_seq_save.select_dtypes(include=["object", "string"]).columns:
    df_seq_save[col] = df_seq_save[col].astype(object)

df_seq_save.to_parquet(SEQUENCES_PATH, index=False, engine="fastparquet")

print(f"Saved {len(df_seq_save):,} rows across {df_seq_save['block_id'].nunique():,} blocks  →  {SEQUENCES_PATH}")
print(f"File size : {SEQUENCES_PATH.stat().st_size / 1_048_576:.1f} MB")

# ── Reload (run this instead of Steps 1‑2 in future sessions) ─────────────────
# df_blocks = pd.read_parquet(SEQUENCES_PATH, engine="fastparquet")
# df_blocks["parameters"] = df_blocks["parameters"].apply(json.loads)
# sequences = {bid: grp.reset_index(drop=True) for bid, grp in df_blocks.groupby("block_id")}
# print(f"Loaded {len(df_blocks):,} rows, {len(sequences):,} blocks")


## Neo4j — Graph Database connection

Graphs are persisted to Neo4j so subsequent runs can skip re-building them.

**Schema written per block:**
- `(:Block {block_id})` — one node per HDFS block  
- `(:EventNode {block_id, cluster_id, template, occurrence_count, …})` — one node per unique template in that block  
- `(Block)-[:HAS_EVENT]->(EventNode)` — membership edge  
- `(EventNode)-[:NEXT_EVENT {weight, time_delta_s, time_delta_std}]->(EventNode)` — temporal transition edge  

**Start the container:**
```bash
# Build (once)
docker build -t hla-neo4j -f infrastructure/neo4j.dockerfile infrastructure/

# Run
docker run -d --name hla-neo4j \
  -p 7474:7474 -p 7687:7687 \
  -v "$(pwd)/data/neo4j:/data" \
  hla-neo4j

# Browser UI → http://localhost:7474  (neo4j / password123)
```


In [ ]:
!pip install neo4j tqdm --quiet


In [ ]:
from neo4j import GraphDatabase

NEO4J_URI      = "bolt://localhost:7687"
NEO4J_USER     = "neo4j"
NEO4J_PASSWORD = "password123"   # matches NEO4J_AUTH in the Dockerfile

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

# Verify connectivity and create indexes once
with driver.session() as session:
    session.run("RETURN 1")   # connection check
    session.run("CREATE INDEX block_id_idx IF NOT EXISTS FOR (b:Block)     ON (b.block_id)")
    session.run("CREATE INDEX event_node_idx IF NOT EXISTS FOR (e:EventNode) ON (e.block_id, e.cluster_id)")

print("✓ Connected to Neo4j and indexes are ready.")
print(f"  URI  : {NEO4J_URI}")
print(f"  User : {NEO4J_USER}")


## Step 3 — Build sequence graphs with node embeddings

For each block sequence, construct a directed `networkx.DiGraph`:
- **Nodes** = unique `(cluster_id, template)` pairs that appear in the sequence  
  - Node features: TF-IDF / bag-of-words embedding of the template string + numeric parameter statistics (count, mean of numeric params)
- **Edges** = directed temporal transitions between consecutive events `(event_i → event_{i+1})`

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# --- Build template embeddings (one vector per unique cluster) ---
# Fit TF-IDF over all unique templates so every node gets a fixed-size feature vector
cluster_to_template = df[["cluster_id", "template"]].drop_duplicates().set_index("cluster_id")["template"]
all_templates = cluster_to_template.tolist()
all_cluster_ids = cluster_to_template.index.tolist()

vectorizer = TfidfVectorizer(analyzer="word", token_pattern=r"[^\s]+")
tfidf_matrix = vectorizer.fit_transform(all_templates)  # shape: (n_clusters, vocab)

# Map cluster_id -> embedding vector (dense numpy array)
cluster_embeddings = {
    cid: tfidf_matrix[i].toarray().flatten()
    for i, cid in enumerate(all_cluster_ids)
}

print(f"Template vocabulary size : {len(vectorizer.vocabulary_)}")
print(f"Embedding dimension      : {tfidf_matrix.shape[1]}")
print(f"Unique clusters          : {len(cluster_embeddings)}")


In [ ]:
import time
from collections import Counter, defaultdict
from tqdm.notebook import tqdm


# ── Graph construction helpers ────────────────────────────────────────────────

def extract_numeric_params(params: list) -> dict:
    """Return basic stats of numeric values found in the parameter list."""
    nums = []
    for p in params:
        try:
            nums.append(float(p))
        except (ValueError, TypeError):
            pass
    if not nums:
        return {"param_count": len(params), "param_num_mean": 0.0, "param_num_max": 0.0}
    return {
        "param_count":    len(params),
        "param_num_mean": float(np.mean(nums)),
        "param_num_max":  float(np.max(nums)),
    }


def build_sequence_graph(seq: pd.DataFrame) -> nx.DiGraph:
    """Build a directed graph from one block's ordered event sequence (optimised)."""
    G = nx.DiGraph()
    G.graph["block_id"] = seq["block_id"].iloc[0]

    cids       = seq["cluster_id"].tolist()
    params_col = seq["parameters"].tolist()
    timestamps = seq["timestamp"].tolist()

    node_params: dict = defaultdict(list)
    node_count:  dict = Counter(cids)

    for cid, params in zip(cids, params_col):
        node_params[cid].extend(params)

    for cid in node_count:
        param_stats = extract_numeric_params(node_params[cid])
        G.add_node(
            cid,
            template=cluster_to_template.get(cid, ""),
            embedding=cluster_embeddings.get(cid, np.zeros(tfidf_matrix.shape[1])),
            occurrence_count=node_count[cid],
            **param_stats,
        )

    edge_deltas: dict = defaultdict(list)
    for i in range(len(cids) - 1):
        src, dst = cids[i], cids[i + 1]
        if G.has_edge(src, dst):
            G[src][dst]["weight"] += 1
        else:
            G.add_edge(src, dst, weight=1, time_delta_s=None)
        t0, t1 = timestamps[i], timestamps[i + 1]
        if t0 is not None and t1 is not None:
            edge_deltas[(src, dst)].append((t1 - t0).total_seconds())

    for (src, dst), deltas in edge_deltas.items():
        G[src][dst]["time_delta_s"]   = float(np.mean(deltas))
        G[src][dst]["time_delta_std"] = float(np.std(deltas))

    for src, dst in G.edges():
        if G[src][dst]["time_delta_s"] is None:
            G[src][dst]["time_delta_s"] = -1.0

    return G


# ── Neo4j batch writer ────────────────────────────────────────────────────────

def write_batch_to_neo4j(batch: dict, session) -> None:
    """Persist a batch of nx.DiGraph objects to Neo4j in one transaction."""
    blocks_data, nodes_data, edges_data = [], [], []

    for bid, G in batch.items():
        blocks_data.append({"block_id": bid})

        for cid, attrs in G.nodes(data=True):
            nodes_data.append({
                "block_id":        bid,
                "cluster_id":      int(cid),
                "template":        attrs.get("template", ""),
                "occurrence_count":int(attrs.get("occurrence_count", 0)),
                "param_count":     int(attrs.get("param_count", 0)),
                "param_num_mean":  float(attrs.get("param_num_mean", 0.0)),
                "param_num_max":   float(attrs.get("param_num_max", 0.0)),
                "embedding":       attrs.get("embedding", np.array([])).tolist(),
            })

        for src, dst, attrs in G.edges(data=True):
            edges_data.append({
                "block_id":      bid,
                "src":           int(src),
                "dst":           int(dst),
                "weight":        int(attrs.get("weight", 1)),
                "time_delta_s":  float(attrs.get("time_delta_s", -1.0)),
                "time_delta_std":float(attrs.get("time_delta_std", 0.0)),
            })

    # 1. Create Block nodes
    session.run(
        "UNWIND $blocks AS b MERGE (:Block {block_id: b.block_id})",
        blocks=blocks_data,
    )
    # 2. Create EventNode nodes and link to Block
    session.run("""
        UNWIND $nodes AS n
        MERGE (e:EventNode {block_id: n.block_id, cluster_id: n.cluster_id})
        SET e.template        = n.template,
            e.occurrence_count = n.occurrence_count,
            e.param_count     = n.param_count,
            e.param_num_mean  = n.param_num_mean,
            e.param_num_max   = n.param_num_max,
            e.embedding       = n.embedding
        WITH e, n
        MATCH (b:Block {block_id: n.block_id})
        MERGE (b)-[:HAS_EVENT]->(e)
    """, nodes=nodes_data)
    # 3. Create NEXT_EVENT transition edges
    session.run("""
        UNWIND $edges AS edge
        MATCH (src:EventNode {block_id: edge.block_id, cluster_id: edge.src})
        MATCH (dst:EventNode {block_id: edge.block_id, cluster_id: edge.dst})
        MERGE (src)-[r:NEXT_EVENT]->(dst)
        SET r.weight        = edge.weight,
            r.time_delta_s  = edge.time_delta_s,
            r.time_delta_std = edge.time_delta_std
    """, edges=edges_data)


# ── Main build loop — batched, with progress bar and Neo4j persistence ────────

BATCH_SIZE  = 100
all_bids    = list(sequences.keys())
graphs: dict = {}

t_start = time.time()

with driver.session() as session:
    for batch_start in tqdm(range(0, len(all_bids), BATCH_SIZE),
                             desc="Building & saving graph batches",
                             unit="batch"):
        batch_bids  = all_bids[batch_start : batch_start + BATCH_SIZE]
        batch_graphs = {bid: build_sequence_graph(sequences[bid]) for bid in batch_bids}
        graphs.update(batch_graphs)
        write_batch_to_neo4j(batch_graphs, session)

elapsed = time.time() - t_start
sample_bid = all_bids[0]
g = graphs[sample_bid]

print(f"\nBuilt & saved {len(graphs):,} graphs in {elapsed:.1f}s "
      f"({elapsed / len(graphs) * 1000:.2f} ms/graph)")
print(f"Sample block : {sample_bid}")
print(f"  Nodes      : {g.number_of_nodes()}")
print(f"  Edges      : {g.number_of_edges()}")
print(f"  Node attrs : {list(list(g.nodes(data=True))[0][1].keys())}")


## Step 4 — Verify time-delta edge weights

Time-delta computation is now folded into `build_sequence_graph` (single pass over each sequence).  
For every directed edge `(A → B)`, the mean and std of elapsed seconds across all consecutive A→B transitions are stored as `time_delta_s` and `time_delta_std`.


In [ ]:
# Time-delta weights are now computed inside build_sequence_graph (single pass).
# This cell verifies the edge attributes on the sample block.

g = graphs[sample_bid]
print(f"Edge attributes for block {sample_bid}:")
for src, dst, attrs in g.edges(data=True):
    tmpl_src = cluster_to_template.get(src, src)[:60]
    tmpl_dst = cluster_to_template.get(dst, dst)[:60]
    print(f"  {tmpl_src!r}  →  {tmpl_dst!r}")
    print(f"    weight={attrs['weight']}  time_delta_s={attrs['time_delta_s']:.2f}s")


In [ ]:
def plot_block_graph(G: nx.DiGraph, block_id: str) -> None:
    pos = nx.spring_layout(G, seed=42)

    node_labels = {
        n: cluster_to_template.get(n, str(n)).split(":")[-1].strip()[:30]
        for n in G.nodes()
    }
    edge_labels = {
        (s, d): f"{attrs['weight']}x\n{attrs['time_delta_s']:.1f}s"
        for s, d, attrs in G.edges(data=True)
    }
    node_sizes = [G.nodes[n].get("occurrence_count", 1) * 300 for n in G.nodes()]

    plt.figure(figsize=(12, 7))
    nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color="steelblue", alpha=0.85)
    nx.draw_networkx_labels(G, pos, labels=node_labels, font_size=7)
    nx.draw_networkx_edges(G, pos, arrows=True, arrowsize=20, edge_color="gray", width=1.5)
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=6)
    plt.title(f"Block sequence graph — {block_id}", fontsize=12)
    plt.axis("off")
    plt.tight_layout()
    plt.show()


plot_block_graph(graphs[sample_bid], sample_bid)


In [ ]:
def display_block_sequence(bid: str, sequences: dict, df_blocks: pd.DataFrame) -> None:
	"""
	Display the full ordered log sequence for a given block_id,
	showing index, timestamp, cluster_id, template, parameters, and raw text.
	"""
	if bid not in sequences:
		print(f"Block ID '{bid}' not found in sequences.")
		return

	seq = sequences[bid]
	print(f"{'='*100}")
	print(f"Block ID : {bid}")
	print(f"Total events: {len(seq)}")
	print(f"{'='*100}\n")

	for i, row in seq.iterrows():
		template_full = cluster_to_template.get(row["cluster_id"], "N/A")
		print(f"[{i:>3}] Timestamp  : {row['timestamp']}")
		print(f"       Cluster ID : {row['cluster_id']}")
		print(f"       Template   : {template_full}")
		print(f"       Parameters : {row['parameters']}")
		print(f"       Raw        : {row['raw']}")
		print(f"       {'-'*90}")


# Display the sample block sequence
display_block_sequence(sample_bid, sequences, df_blocks)

## Step 5 — Retrieve & visualise sample graphs from Neo4j

Query the database for a handful of persisted block graphs, reconstruct them as `networkx.DiGraph`
objects, then visualise and validate them against the in-memory `graphs` dict.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# ── 1. Fetch N sample blocks from Neo4j ──────────────────────────────────────
N_SAMPLES = 5   # change as needed

FETCH_BLOCKS_Q = """
MATCH (b:Block)
RETURN b.block_id AS block_id
LIMIT $n
"""

FETCH_GRAPH_Q = """
MATCH (b:Block {block_id: $bid})-[:HAS_EVENT]->(e:EventNode)
OPTIONAL MATCH (e)-[r:NEXT_EVENT]->(e2:EventNode {block_id: $bid})
RETURN
  e.cluster_id      AS src_cid,
  e.template        AS src_tmpl,
  e.occurrence_count AS src_occ,
  e.param_count     AS src_pc,
  e.param_num_mean  AS src_mean,
  e.param_num_max   AS src_max,
  e2.cluster_id     AS dst_cid,
  r.weight          AS weight,
  r.time_delta_s    AS time_delta_s,
  r.time_delta_std  AS time_delta_std
"""

def reconstruct_graph_from_neo4j(bid: str, session) -> nx.DiGraph:
    """Reconstruct a networkx DiGraph from persisted Neo4j data."""
    G = nx.DiGraph()
    G.graph["block_id"] = bid

    rows = session.run(FETCH_GRAPH_Q, bid=bid).data()
    for row in rows:
        src = row["src_cid"]
        if src is None:
            continue
        if not G.has_node(src):
            G.add_node(
                src,
                template=row["src_tmpl"] or "",
                occurrence_count=row["src_occ"] or 1,
                param_count=row["src_pc"] or 0,
                param_num_mean=row["src_mean"] or 0.0,
                param_num_max=row["src_max"] or 0.0,
            )
        dst = row["dst_cid"]
        if dst is not None:
            if not G.has_node(dst):
                # minimal node entry; full attrs will be filled when its src row arrives
                G.add_node(dst)
            if not G.has_edge(src, dst):
                G.add_edge(
                    src, dst,
                    weight=row["weight"] or 1,
                    time_delta_s=row["time_delta_s"] if row["time_delta_s"] is not None else -1.0,
                    time_delta_std=row["time_delta_std"] or 0.0,
                )
    return G


with driver.session() as _session:
    sample_bids_neo4j = [r["block_id"] for r in _session.run(FETCH_BLOCKS_Q, n=N_SAMPLES).data()]
    neo4j_graphs = {bid: reconstruct_graph_from_neo4j(bid, _session) for bid in sample_bids_neo4j}

print(f"Retrieved {len(neo4j_graphs)} block graphs from Neo4j:")
for bid, G in neo4j_graphs.items():
    print(f"  {bid}  →  {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")


# ── 2. Validate against in-memory graphs (if available) ──────────────────────
print("\n── Validation vs in-memory graphs ──")
_graphs_in_memory = "graphs" in dir() or "graphs" in vars()
if not _graphs_in_memory:
    print("  'graphs' dict not in scope – run the build cell first for full validation.")
any_mismatch = False
_graphs_ref = graphs if _graphs_in_memory else {}  # type: ignore[name-defined]
for bid, G_neo4j in neo4j_graphs.items():
    if bid not in _graphs_ref:
        print(f"  [{bid}] not in memory – skipping")
        continue
    G_mem = graphs[bid]
    n_ok  = G_neo4j.number_of_nodes() == G_mem.number_of_nodes()
    e_ok  = G_neo4j.number_of_edges() == G_mem.number_of_edges()
    status = "✓" if (n_ok and e_ok) else "✗"
    if not (n_ok and e_ok):
        any_mismatch = True
    print(
        f"  {status} {bid[:30]}  "
        f"nodes: neo4j={G_neo4j.number_of_nodes()} mem={G_mem.number_of_nodes()}  "
        f"edges: neo4j={G_neo4j.number_of_edges()} mem={G_mem.number_of_edges()}"
    )
if not any_mismatch:
    print("  All graphs match ✓")


# ── 3. Visualise the retrieved graphs ────────────────────────────────────────
def plot_neo4j_graph(G: nx.DiGraph, block_id: str) -> None:
    """Plot a graph reconstructed from Neo4j (uses template attr directly)."""
    pos = nx.spring_layout(G, seed=42)

    node_labels = {
        n: (G.nodes[n].get("template", str(n)) or str(n)).split(":")[-1].strip()[:28]
        for n in G.nodes()
    }
    edge_labels = {
        (s, d): f"{attrs.get('weight', '')}x\n{attrs.get('time_delta_s', 0):.1f}s"
        for s, d, attrs in G.edges(data=True)
    }
    node_sizes = [max(G.nodes[n].get("occurrence_count", 1), 1) * 300 for n in G.nodes()]

    plt.figure(figsize=(12, 6))
    nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color="darkorange", alpha=0.85)
    nx.draw_networkx_labels(G, pos, labels=node_labels, font_size=7, font_color="white")
    nx.draw_networkx_edges(G, pos, arrows=True, arrowsize=20, edge_color="#444", width=1.5)
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=6)
    plt.title(f"[Neo4j] Block graph — {block_id}", fontsize=11)
    plt.axis("off")
    plt.tight_layout()
    plt.show()


for bid, G in neo4j_graphs.items():
    plot_neo4j_graph(G, bid)


In [ ]:

# ── Diagnostic: why 23 log lines → fewer graph nodes ─────────────────────────
BID = "blk_-1000083860370843431"

if BID not in sequences:
    print(f"Block {BID} not found – check the MAX_LINES cap.")
else:
    seq = sequences[BID]
    G   = graphs[BID]

    print(f"Raw log lines (df_blocks rows) : {len(seq)}")
    print(f"Unique cluster_ids (graph nodes): {G.number_of_nodes()}")
    print(f"Graph edges                     : {G.number_of_edges()}")
    print()

    # Show every cluster_id and how many times it fires in this sequence
    counts = seq["cluster_id"].value_counts().reset_index()
    counts.columns = ["cluster_id", "occurrences"]
    counts["template"] = counts["cluster_id"].map(cluster_to_template).str[:80]
    print("Cluster firing counts (sorted by occurrences desc):")
    print(counts.to_string(index=False))
    print()

    # Highlight repeated clusters – these are the lines 'lost' to node collapsing
    repeated = counts[counts["occurrences"] > 1]
    extra_lines = (repeated["occurrences"] - 1).sum()
    print(f"Clusters that fire > 1 time     : {len(repeated)}")
    print(f"Extra lines collapsed into nodes : {extra_lines}")
    print(f"  → {len(seq)} log lines  –  {extra_lines} collapsed  =  {G.number_of_nodes()} nodes  ✓")
    print()

    # Walk the full ordered sequence to show every firing
    print("Full ordered sequence (index | cluster_id | occurrences_so_far | template):")
    seen: dict = {}
    for i, row in seq.iterrows():
        cid = row["cluster_id"]
        seen[cid] = seen.get(cid, 0) + 1
        tmpl = cluster_to_template.get(cid, "N/A")[:70]
        print(f"  [{i:>2}] cid={cid}  fire#{seen[cid]}  {tmpl}")


In [ ]:

# ── Root-cause: scan raw log for ALL occurrences of the block ─────────────────
import re
from pathlib import Path

BID = "blk_-1000083860370843431"
BLOCK_RE = re.compile(r"blk_-?\d+")

raw_lines = []
with Path(LOG_PATH).open("r", errors="replace") as fh:
    for i, raw in enumerate(fh, start=1):
        line = raw.rstrip("\n")
        if BID in line:
            raw_lines.append((i, line))

print(f"Raw log lines containing {BID}: {len(raw_lines)}")
print(f"Lines inside MAX_LINES cap ({MAX_LINES:,}): {sum(1 for ln_no, _ in raw_lines if ln_no <= MAX_LINES)}")
print(f"Lines BEYOND MAX_LINES cap            : {sum(1 for ln_no, _ in raw_lines if ln_no > MAX_LINES)}")
print()

# Try matching each raw line against Drain's final templates
matched, unmatched = [], []
for ln_no, line in raw_lines:
    m = parser.miner.match(line)
    if m is None:
        unmatched.append((ln_no, line))
    else:
        matched.append((ln_no, line, m.cluster_id))

print(f"Matched by Drain  : {len(matched)}")
print(f"Unmatched by Drain: {len(unmatched)}")

if unmatched:
    print("\nUnmatched lines (these are silently dropped in annotate_file):")
    for ln_no, line in unmatched:
        print(f"  line {ln_no:>7}: {line[:120]}")
